# Higher-Order Functions

Goal: browse the different ways Python treats functions as first-class values — passing functions as arguments, returning functions from functions, the built-in HOFs (`map`/`filter`/`sorted`/`reduce`), composing functions, decorators, `functools.partial`, and combining several of these together.

## 1. Functions are first-class objects

A function can be assigned to a variable, stored in a list, or passed around like any other value — that's what makes higher-order functions possible.

In [ ]:
def shout(text):
    return text.upper() + "!"

# functions are first-class: assign one to a variable, pass it around like any value
yell = shout
print(yell("hello"))
print(callable(yell))

funcs = [shout, str.lower, str.title]
for f in funcs:
    print(f("Python Rocks"))

HELLO!
True
PYTHON ROCKS!
python rocks
Python Rocks


## 2. Passing a function as an argument

A higher-order function is simply one that accepts another function as a parameter (and/or returns one). `apply_twice` doesn't care what `func` does — it just calls it.

In [ ]:
def apply_twice(func, value):
    return func(func(value))  # a HOF: takes another function as an argument

def increment(n):
    return n + 1

print(apply_twice(increment, 5))          # 5 -> 6 -> 7
print(apply_twice(str.upper, "ab"))       # 'AB' -> 'AB' (idempotent here)
print(apply_twice(lambda s: s + "!", "hi"))  # 'hi!' -> 'hi!!'

7
AB
hi!!


## 3. Returning a function (function factory)

A function can build and return another function, customized by the arguments it was called with — the returned function is a closure over those arguments.

In [ ]:
def make_power(exponent):
    def power(base):
        return base ** exponent
    return power  # a HOF: returns a new function

square = make_power(2)
cube = make_power(3)

print(square(5))
print(cube(2))
print([make_power(2)(n) for n in range(1, 5)])

25
8
[1, 4, 9, 16]


## 4. Built-in higher-order functions

`map`, `filter`, `sorted`, and `max`/`min` all accept a function to customize their behavior — the standard library's own HOFs.

In [ ]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]

print(list(map(lambda n: n * 10, numbers)))
print(list(filter(lambda n: n > 3, numbers)))
print(sorted(numbers, key=lambda n: -n))
print(max(numbers, key=lambda n: -n))  # "largest" by negated value == smallest

[30, 10, 40, 10, 50, 90, 20, 60]
[4, 5, 9, 6]
[9, 6, 5, 4, 3, 2, 1, 1]
1


## 5. Composing functions

A HOF can combine several functions into one new function that applies them in sequence — this is how function "pipelines" are built.

In [ ]:
def compose(*funcs):
    def composed(x):
        result = x
        for f in reversed(funcs):  # apply right-to-left, like math composition f(g(x))
            result = f(result)
        return result
    return composed

def double(n):
    return n * 2

def increment(n):
    return n + 1

double_then_increment = compose(increment, double)  # increment(double(x))
print(double_then_increment(5))  # double(5)=10, increment(10)=11

pipeline = compose(str, double, increment)  # str(double(increment(x)))
print(pipeline(3))  # increment(3)=4, double(4)=8, str(8)='8'

11
8


## 6. Decorators — functions that wrap functions

A decorator is a HOF that takes a function and returns a new function wrapping it with extra behavior. `@logged` is shorthand for `add = logged(add)`.

In [ ]:
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper  # a decorator: HOF that wraps a function with extra behavior

@logged
def add(a, b):
    return a + b

print(add(2, 3))

calling add(2, 3)
add returned 5
5


## 7. `functools.partial`

`partial` is a built-in HOF that takes a function and pre-fills some of its arguments, returning a new, more specific function.

In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)  # HOF that pre-fills an argument
cube = partial(power, exponent=3)

print(square(5))
print(cube(2))
print(list(map(partial(power, exponent=2), [1, 2, 3, 4])))

25
8
[1, 4, 9, 16]


## 8. Multiple function parameters at once

A HOF isn't limited to accepting a single function — `process` takes both a `transform` and a `predicate`, combining them in one pass.

In [ ]:
def process(items, transform, predicate):
    return [transform(item) for item in items if predicate(item)]  # two functions in, one HOF

words = ["apple", "kiwi", "banana", "fig", "cherry"]

result = process(
    words,
    transform=str.upper,
    predicate=lambda w: len(w) > 4,
)
print(result)

['APPLE', 'BANANA', 'CHERRY']


## 9. Combining several HOFs together

Real code often chains multiple higher-order functions — here `reduce` builds up a running maximum, and `map`/`filter`/`sum` combine to compute an aggregate without writing an explicit loop.

In [ ]:
from functools import reduce

def running_max(history, current):
    if not history:
        return [current]
    return history + [max(history[-1], current)]

numbers = [3, 1, 4, 1, 5, 9, 2, 6]
maxes = reduce(running_max, numbers, [])
print(maxes)  # running maximum after each element

# sum of squares of even numbers, all via HOFs, no loop written out
result = sum(map(lambda n: n * n, filter(lambda n: n % 2 == 0, numbers)))
print(result)

[3, 3, 4, 4, 5, 9, 9, 9]
56
